# Test the Iceberg MCP server connection

Runs inside a **CML Session in the `iceberg-mcp` project**. Connects to the deployed MCP
Application over **streamable HTTP** (`/mcp/`) and exercises `get_schema()` and `execute_query(...)`.

**Credentials are passed from this notebook**, per connection, as HTTP headers — nothing secret
lives on the server:
- `Authorization: Bearer <CML key>` authenticates to the CML/Knox gateway.
- `X-Impala-User` / `X-Impala-Password` (+ optional `X-Impala-Host` / `X-Impala-Database`) are read
  per request by the server and used to open the Impala connection.

A successful `get_schema()` proves the whole path: notebook → Knox → MCP app → impyla → Impala.

In [ ]:
# fastmcp is installed in the project's ~/.local by the bootstrap job; install here too if needed.
try:
    import fastmcp  # noqa
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'fastmcp>=2.9.2'])
import fastmcp
print('fastmcp', fastmcp.__version__)

In [ ]:
import os, getpass

# Deployed app base URL (no trailing /mcp/). Override via MCP_APP_URL env if needed.
APP_URL = os.environ.get(
    'MCP_APP_URL',
    'https://icebergmcp2.ml-cc27e256-5e1.se-sandb.a465-9q4k.cloudera.site',
)
MCP_ENDPOINT = APP_URL.rstrip('/') + '/mcp/'

# Gateway auth. In a CML session this is provided as CDSW_APIV2_KEY; else export CML_API_KEY.
TOKEN = os.environ.get('CDSW_APIV2_KEY') or os.environ.get('CML_API_KEY')
assert TOKEN, 'No Bearer token: set CDSW_APIV2_KEY (auto in CML) or CML_API_KEY'

# Impala credentials — prompted here, passed as headers, never stored.
IMPALA_USER = os.environ.get('IMPALA_USER') or (input('Impala user [avelloal]: ') or 'avelloal')
IMPALA_PASSWORD = os.environ.get('IMPALA_PASSWORD') or getpass.getpass('Impala (workload) password: ')
IMPALA_HOST = os.environ.get('IMPALA_HOST', 'coordinator-xom-ice-kudu.dw-se-sandbox-aws.a465-9q4k.cloudera.site')
IMPALA_DATABASE = os.environ.get('IMPALA_DATABASE', 'default')

HEADERS = {
    'Authorization': f'Bearer {TOKEN}',
    'X-Impala-User': IMPALA_USER,
    'X-Impala-Password': IMPALA_PASSWORD,
    'X-Impala-Host': IMPALA_HOST,
    'X-Impala-Database': IMPALA_DATABASE,
}
print('endpoint:', MCP_ENDPOINT, '| impala user:', IMPALA_USER, '| db:', IMPALA_DATABASE)

In [ ]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

transport = StreamableHttpTransport(url=MCP_ENDPOINT, headers=HEADERS)

async with Client(transport) as client:
    tools = await client.list_tools()
    print('tools:', [t.name for t in tools])

In [ ]:
# get_schema() -> JSON list of table names in IMPALA_DATABASE (proves Impala connectivity)
async with Client(transport) as client:
    schema = await client.call_tool('get_schema', {})
    print(schema.data if hasattr(schema, 'data') else schema)

In [ ]:
# execute_query(): read-only only (select/show/describe/with). SHOW DATABASES is a safe smoke test.
async with Client(transport) as client:
    res = await client.call_tool('execute_query', {'query': 'SHOW DATABASES'})
    print(res.data if hasattr(res, 'data') else res)